In [32]:
!pip install -q duckdb beautifulsoup4 lxml requests pyarrow pandas

In [33]:
import os, re, json, time, zlib, queue, threading, collections
from pathlib import Path
import urllib.request
import pandas as pd

H-2. Persiapan: Unduh Data dan Jalankan Portal Lokal

In [34]:
URL_DATA = ("https://raw.githubusercontent.com/ibamibrahim/"
            "dataset-judul-berita-indonesia/master/detik_news_title.csv")

if not os.path.exists("detik_news_title.csv"):
    urllib.request.urlretrieve(URL_DATA, "detik_news_title.csv")

df_sumber = pd.read_csv("detik_news_title.csv")
print("Ukuran berkas :", round(os.path.getsize("detik_news_title.csv") / 1e6, 1), "MB")
print("Baris, kolom :", df_sumber.shape)
print("Kolom :", list(df_sumber.columns))

import http.server, urllib.parse, html

HALAMAN_UKURAN = 100  # judul per halaman

def jalankan_portal(df, port=8000):
    """Portal berita lokal (siap pakai): menyajikan judul dalam halaman HTML."""
    baris = df.to_dict("records")
    n_hal = (len(baris) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN

    class Handler(http.server.BaseHTTPRequestHandler):
        def log_message(self, *args):  # matikan log
            pass

        def _kirim(self, kode, teks, tipe="text/html; charset=utf-8"):
            data = teks.encode("utf-8")
            self.send_response(kode)
            self.send_header("Content-Type", tipe)
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)

        def do_GET(self):
            u = urllib.parse.urlparse(self.path)
            if u.path == "/robots.txt":
                self._kirim(200, "User-agent: *\nDisallow: /arsip-privat/\n", "text/plain")
            elif u.path == "/arsip-privat/":
                self._kirim(200, "<p>Halaman ini tidak boleh di-crawl.</p>")
            elif u.path == "/berita":
                q = urllib.parse.parse_qs(u.query)
                hal = int(q.get("hal", ["1"])[0])
                if not 1 <= hal <= n_hal:
                    return self._kirim(404, "<p>Halaman tidak ada</p>")
                potong = baris[(hal - 1) * HALAMAN_UKURAN: hal * HALAMAN_UKURAN]
                isi = []
                for r in potong:
                    iso = pd.to_datetime(r["date"], format="%m/%d/%Y").strftime("%Y-%m-%d")
                    isi.append(
                        '<article class="item">'
                        f'<h2 class="judul"><a href="{html.escape(r["url"])}">{html.escape(r["title"])}</a></h2>'
                        f'<span class="kategori">{html.escape(r["category"])}</span>'
                        f'<time datetime="{iso}">{html.escape(r["date"])}</time>'
                        '</article>')
                self._kirim(200, f"<html><body><h1>Portal Contoh</h1>{''.join(isi)}"
                                 f'<nav data-total="{n_hal}"></nav></body></html>')
            else:
                self._kirim(404, "<p>Tidak ditemukan</p>")

    http.server.ThreadingHTTPServer.allow_reuse_address = True
    server = http.server.ThreadingHTTPServer(("127.0.0.1", port), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, n_hal


if "server" in globals():
    server.shutdown(); server.server_close()

server, N_HALAMAN = jalankan_portal(df_sumber)
BASE = "http://127.0.0.1:8000"
print("Portal lokal aktif di", BASE, "| jumlah halaman:", N_HALAMAN)

Ukuran berkas : 16.1 MB
Baris, kolom : (91017, 4)
Kolom : ['date', 'url', 'title', 'category']
Portal lokal aktif di http://127.0.0.1:8000 | jumlah halaman: 911


H-3. Crawling Portal Lokal: Latihan Aman

In [35]:
import requests, urllib.robotparser
from bs4 import BeautifulSoup

UA = "KelasBigData-UNAND/1.0 (praktikum; kontak: dosen)"

def baca_robots(base):
    r = requests.get(base + "/robots.txt", headers={"User-Agent": UA}, timeout=10)
    rp = urllib.robotparser.RobotFileParser()
    rp.parse(r.text.splitlines())
    return rp

robots = baca_robots(BASE)
for jalur in ["/berita?hal=1", "/arsip-privat/"]:
    boleh = robots.can_fetch(UA, BASE + jalur)
    print(f"{jalur:<18} boleh di-crawl? {boleh}")

def parse_halaman(teks_html):
    """Ubah HTML satu halaman menjadi daftar dict (judul, url, kategori, tanggal)."""
    sup = BeautifulSoup(teks_html, "html.parser")
    hasil = []
    for a in sup.select("article.item"):
        link = a.select_one("h2.judul a")
        hasil.append({
            "judul": link.get_text(strip=True),
            "url": link["href"],
            "kategori": a.select_one("span.kategori").get_text(strip=True),
            "tanggal": a.select_one("time")["datetime"],
        })
    return hasil

def crawl_iter(base, halaman, delay=0.0, ua=UA):
    """Siap pakai: ambil halaman satu per satu, patuhi robots.txt, beri jeda."""
    rp = baca_robots(base)
    sesi = requests.Session()
    sesi.headers["User-Agent"] = ua
    for h in halaman:
        url = f"{base}/berita?hal={h}"
        if not rp.can_fetch(ua, url):
            print("Dilewati (robots.txt):", url)
            continue
        for percobaan in range(3):
            try:
                r = sesi.get(url, timeout=10)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                time.sleep(0.5 * (percobaan + 1))
        else:
            print("Gagal setelah 3 percobaan:", url)
            continue
        yield from parse_halaman(r.text)
        time.sleep(delay)

contoh = list(crawl_iter(BASE, range(1, 4)))
print("Judul dari 3 halaman:", len(contoh))
for c in contoh[:3]:
    print(c)

/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False
Judul dari 3 halaman: 300
{'judul': 'Kemnaker Awasi TKA di Meikarta', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916114/kemnaker-awasi-tka-di-meikarta', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'BNI Digitalkan BNI Java Jazz 2020', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916199/bni-digitalkan-bni-java-jazz-2020', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'Terbang ke Australia, Edhy Prabowo Mau Genjot Budi Daya Lobster', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916193/terbang-ke-australia-edhy-prabowo-mau-genjot-budi-daya-lobster', 'kategori': 'finance', 'tanggal': '2020-02-26'}


H-4. Crawling Berita Sungguhan dari Sitemap detik.com

Langkah 1 — parser sitemap

In [36]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse

POLA_ARTIKEL = re.compile(r"https?://([\w-]+)\.detik\.com/.+/d-(\d+)/")

def parse_sitemap_news(xml_data):
    """Ubah sitemap berita (XML) menjadi daftar dict (judul, url, kategori, id, tanggal, kata_kunci)."""
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []
    for u in sup.find_all("url"):
        loc, judul, waktu = u.find("loc"), u.find("title"), u.find("publication_date")
        if loc is None or judul is None:
            continue
        url = loc.get_text(strip=True)
        m = POLA_ARTIKEL.match(url)
        kunci = u.find("keywords")
        hasil.append({
            "judul": re.sub(r"\s+", " ", judul.get_text(strip=True)),
            "url": url,
            "kategori": m.group(1) if m else urlparse(url).netloc.split(".")[0],  # news, inet, sport, ...
            "id": int(m.group(2)) if m else None,
            "tanggal": waktu.get_text(strip=True)[:10] if waktu else None,
            "kata_kunci": kunci.get_text(strip=True) if kunci else "",
        })
    return hasil

Langkah 2 — uji tanpa internet.

In [37]:
# Contoh XML: bentuk disalin dari sitemap berita asli (tiga entri hasil uji dosen, 29 Okt 2026).
CONTOH_XML = """<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9"
        xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729282/mk-kepala-daerah-terdakwa-politik-tak-bisa-diberhentikan-tanpa-usulan-dprd]]></loc>
    <news:news>
      <news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
      <news:publication_date>2026-10-29T12:52:00+00:00</news:publication_date>
      <news:title><![CDATA[MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD]]></news:title>
      <news:keywords><![CDATA[mahkamah konstitusi,mk,hukum,kepala daerah]]></news:keywords>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729280/tep-unair-periksa-1-000-sapi-di-manokwari-lewat-posyandu-ternak-keliling]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:50:00+00:00</news:publication_date>
      <news:title><![CDATA[TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling]]></news:title>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729264/andre-rosiade-dorong-sedimen-sungai-akibat-banjir-jadi-sumber-pendapatan-padang]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:44:00+00:00</news:publication_date>
      <news:title><![CDATA[Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang]]></news:title>
    </news:news>
  </url>
</urlset>"""

for x in parse_sitemap_news(CONTOH_XML):
    print({k: x[k] for k in ("judul", "kategori", "id", "tanggal")})
print("kata kunci entri pertama:", parse_sitemap_news(CONTOH_XML)[0]["kata_kunci"])
print("Sama bila masukan berupa bytes:",
      parse_sitemap_news(CONTOH_XML.encode("utf-8")) == parse_sitemap_news(CONTOH_XML))

{'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'kategori': 'news', 'id': 8729282, 'tanggal': '2026-10-29'}
{'judul': 'TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling', 'kategori': 'news', 'id': 8729280, 'tanggal': '2026-10-29'}
{'judul': 'Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang', 'kategori': 'news', 'id': 8729264, 'tanggal': '2026-10-29'}
kata kunci entri pertama: mahkamah konstitusi,mk,hukum,kepala daerah
Sama bila masukan berupa bytes: True


Langkah 3 — fungsi pengambilan yang sopan.

In [38]:
import urllib.robotparser, socket
socket.setdefaulttimeout(20)  # RobotFileParser.read() tidak punya timeout sendiri

UA_DETIK = "BigData/1.0 (praktikum pembelajaran; kontak: GANTI_DENGAN_EMAIL_ANDA)"
DELAY = 2.0              # detik antar-request. Jangan dikurangi.
BATAS_PER_BAGIAN = 7     # judul yang diambil dari tiap sitemap
BATAS_TOTAL = 20         # batas seluruh crawling
SITEMAP_DIPILIH = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://sport.detik.com/basket/sitemap_news.xml",
]

_robots = {}
def boleh_diambil(url, ua=UA_DETIK):
    """robots.txt berlaku per host: news.detik.com dan inet.detik.com punya berkas sendiri."""
    host = urlparse(url).netloc
    if host not in _robots:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(f"https://{host}/robots.txt")
        rp.read()
        _robots[host] = rp
    return _robots[host].can_fetch(ua, url)

def ambil_xml(url, sesi, delay=DELAY):
    if not boleh_diambil(url):
        raise PermissionError("robots.txt melarang: " + url)
    r = sesi.get(url, timeout=20)
    if r.status_code in (403, 429):
        raise RuntimeError(f"Server membalas {r.status_code}. Berhenti, tunggu, dan kurangi laju.")
    r.raise_for_status()
    time.sleep(delay)
    return r.content  # bytes: pengkodean dibaca dari deklarasi XML

def crawl_detik():
    sesi = requests.Session()
    sesi.headers["User-Agent"] = UA_DETIK
    hasil = []
    for sm in SITEMAP_DIPILIH:
        if len(hasil) >= BATAS_TOTAL:
            break
        hasil += parse_sitemap_news(ambil_xml(sm, sesi))[:BATAS_PER_BAGIAN]
    return hasil[:BATAS_TOTAL]

Langkah 4 — jalankan dan periksa.

In [39]:
try:
    sampel = crawl_detik()
except Exception as e:
    sampel = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if sampel:
    Path("contoh_crawl").mkdir(exist_ok=True)
    with open("contoh_crawl/detik_sitemap.jsonl", "w", encoding="utf-8") as f:
        for x in sampel:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    cek = pd.DataFrame(sampel)
    print("Judul terkumpul :", len(cek))
    print("Judul kosong    :", (cek["judul"].str.strip() == "").sum())
    print("URL duplikat    :", cek["url"].duplicated().sum())
    print("Tanggal kosong  :", cek["tanggal"].isna().sum())
    print("Kategori        :", cek["kategori"].value_counts().to_dict())
    print(cek[["judul", "kategori", "tanggal"]].head(3).to_string(index=False))

Judul terkumpul : 20
Judul kosong    : 0
URL duplikat    : 6
Tanggal kosong  : 0
Kategori        : {'news': 7, '': 7, 'sport': 6}
                                                                    judul kategori    tanggal
Kompolnas Sebut Sidang Etik Ahrie Sonta Sesuai Prosedur dan Akuntabilitas     news 2026-11-10
                    Hidayat Nur Wahid Mundur dari Jabatan Wakil Ketua MPR     news 2026-11-10
     Cicit Jenderal Hoegeng Cerita Isi Obrolan Bareng Prabowo dan Kapolri     news 2026-11-10


H-5. Channel ala Flume ke `lapisan_mentah`

In [40]:
RAW = Path("lapisan_mentah"); RAW.mkdir(exist_ok=True)
for f in RAW.glob("*.jsonl"):
    f.unlink()

channel = queue.Queue(maxsize=2000)
STOP = object()
statistik = {"max_antrean": 0, "masuk": 0, "keluar": 0, "file": 0}

def source():
    for ev in crawl_iter(BASE, range(1, N_HALAMAN + 1)):
        channel.put(ev)  # menunggu bila channel penuh
        statistik["masuk"] += 1
        statistik["max_antrean"] = max(statistik["max_antrean"], channel.qsize())
    channel.put(STOP)

def sink(ukuran_batch=5000):
    batch, nomor = [], 0

    def tulis():
        nonlocal batch, nomor
        nomor += 1
        with open(RAW / f"batch_{nomor:04d}.jsonl", "w", encoding="utf-8") as f:
            for ev in batch:
                f.write(json.dumps(ev, ensure_ascii=False) + "\n")
        statistik["file"] = nomor
        batch = []

    while True:
        ev = channel.get()
        if ev is STOP:
            break
        batch.append(ev); statistik["keluar"] += 1
        if len(batch) >= ukuran_batch:
            tulis()
    if batch:
        tulis()

t0 = time.perf_counter()
t_src = threading.Thread(target=source); t_snk = threading.Thread(target=sink)
t_src.start(); t_snk.start(); t_src.join(); t_snk.join()
waktu = time.perf_counter() - t0

print("Event masuk channel :", f"{statistik['masuk']:,}")
print("Event ditulis sink  :", f"{statistik['keluar']:,}")
print("File batch          :", statistik["file"], "->", sorted(p.name for p in RAW.glob('*.jsonl'))[:3], "...")
print("Antrean maks <= 2000:", statistik["max_antrean"] <= 2000)
print("Total ukuran mentah :", round(sum(p.stat().st_size for p in RAW.glob('*.jsonl')) / 1e6, 1), "MB")
print("Waktu crawl total   :", round(waktu, 1), "detik untuk", N_HALAMAN, "halaman")

Event masuk channel : 91,017
Event ditulis sink  : 91,017
File batch          : 19 -> ['batch_0001.jsonl', 'batch_0002.jsonl', 'batch_0003.jsonl'] ...
Antrean maks <= 2000: True
Total ukuran mentah : 20.7 MB
Waktu crawl total   : 59.4 detik untuk 911 halaman


H-6. Quality, Cleaning, dan Parquet Berpartisi

In [41]:
mentah = pd.concat([pd.read_json(p, lines=True) for p in sorted(RAW.glob("*.jsonl"))],
                   ignore_index=True)
print("Baris mentah:", f"{len(mentah):,}")

KATEGORI_DIKENAL = {"finance", "food", "health", "hot", "inet", "news", "oto", "sport", "travel"}

def aturan_kualitas(d):
    """Setiap aturan menghasilkan Series True/False per baris."""
    jml_kata = d["judul"].fillna("").str.split().str.len()
    return {
        "judul tidak kosong"  : d["judul"].fillna("").str.strip() != "",
        "judul 3-30 kata"     : jml_kata.between(3, 30),
        "kategori dikenal"    : d["kategori"].isin(KATEGORI_DIKENAL),
        "tanggal valid"       : pd.to_datetime(d["tanggal"], format="%Y-%m-%d", errors="coerce").notna(),
        "url artikel valid"   : d["url"].fillna("").str.contains(r"/d-?\d+", regex=True),
        "judul tidak duplikat": ~d["judul"].duplicated(keep=False),
    }

for nama, lulus in aturan_kualitas(mentah).items():
    print(f"{nama:<24} gagal: {(~lulus).sum():>6,} ({(~lulus).mean()*100:5.2f}%)")

Baris mentah: 91,017
judul tidak kosong       gagal:      0 ( 0.00%)
judul 3-30 kata          gagal:     78 ( 0.09%)
kategori dikenal         gagal:      0 ( 0.00%)
tanggal valid            gagal:      0 ( 0.00%)
url artikel valid        gagal:  5,640 ( 6.20%)
judul tidak duplikat     gagal:    235 ( 0.26%)


In [42]:
bersih = mentah.copy()
bersih["judul"] = bersih["judul"].str.replace(r"\s+", " ", regex=True).str.strip()
bersih = bersih.drop_duplicates(subset="judul", keep="first")
bersih = bersih[bersih["judul"].str.split().str.len().between(3, 30)].copy()
bersih["tanggal"] = pd.to_datetime(bersih["tanggal"], format="%Y-%m-%d")
bersih["bulan"] = bersih["tanggal"].dt.strftime("%Y-%m")
bersih["id"] = bersih["url"].str.extract(r"/d-?(\d+)")[0].astype("Int64")  # kosong bila url rusak
bersih["url"] = bersih["url"].where(bersih["url"].str.contains(r"/d-?\d+", regex=True))
print("Baris setelah dibersihkan:", f"{len(bersih):,}", "| dibuang:", f"{len(mentah) - len(bersih):,}")

Baris setelah dibersihkan: 90,818 | dibuang: 199


In [43]:
import shutil, pyarrow.dataset as ds

def simpan_terpartisi(data, kolom_partisi, folder):
    shutil.rmtree(folder, ignore_errors=True)
    data.to_parquet(folder, partition_cols=kolom_partisi, index=False)
    n = len(list(Path(folder).rglob("*.parquet")))
    ukuran = sum(p.stat().st_size for p in Path(folder).rglob("*.parquet")) / 1e6
    return n, ukuran

# BAGIAN KUNCI: bandingkan pilihan kolom partisi
for kolom in (["kategori"], ["bulan"], ["kategori", "bulan"]):
    n, mb = simpan_terpartisi(bersih, kolom, "uji_partisi")
    print(f"partisi {str(kolom):<22} -> {n:>3} file, {mb:5.1f} MB")
shutil.rmtree("uji_partisi")

TERKURASI = "lapisan_terkurasi"
n, mb = simpan_terpartisi(bersih, ["kategori"], TERKURASI)
print("\nDisimpan ke", TERKURASI, "dengan partisi kategori:", n, "file,", round(mb, 1), "MB")

dataset = ds.dataset(TERKURASI, format="parquet", partitioning="hive")
semua = list(dataset.get_fragments())
hanya = list(dataset.get_fragments(filter=ds.field("kategori") == "finance"))
print("Partition pruning: baca", len(hanya), "dari", len(semua), "file untuk kategori = finance")

partisi ['kategori']           ->   9 file,   8.4 MB
partisi ['bulan']              ->   6 file,   8.3 MB
partisi ['kategori', 'bulan']  ->  54 file,   8.4 MB

Disimpan ke lapisan_terkurasi dengan partisi kategori: 9 file, 8.4 MB
Partition pruning: baca 1 dari 9 file untuk kategori = finance


 H-7. Query DuckDB dan Bentuk Dokumen

In [44]:
import duckdb
con = duckdb.connect()
con.sql(f"CREATE VIEW berita AS SELECT * FROM read_parquet('{TERKURASI}/*/*.parquet', hive_partitioning=true)")

print(con.sql("SELECT kategori, COUNT(*) AS jumlah FROM berita GROUP BY kategori ORDER BY jumlah DESC").df().to_string(index=False))
print()
print(con.sql("""
    SELECT bulan, COUNT(*) AS judul_corona
    FROM berita WHERE lower(judul) LIKE '%corona%'
    GROUP BY bulan ORDER BY bulan""").df().to_string(index=False))

dokumen = {
    "id": int(bersih.iloc[0]["id"]),
    "judul": bersih.iloc[0]["judul"],
    "kategori": bersih.iloc[0]["kategori"],
    "tanggal": str(bersih.iloc[0]["tanggal"].date()),
    "kata": re.findall(r"[a-z]+", bersih.iloc[0]["judul"].lower()),
}
print()
print(json.dumps(dokumen, ensure_ascii=False, indent=2))

kategori  jumlah
    news   32321
     hot   16290
 finance   14141
  travel    6453
    inet    5577
  health    4914
     oto    4375
    food    4314
   sport    2433

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

{
  "id": 4916114,
  "judul": "Kemnaker Awasi TKA di Meikarta",
  "kategori": "finance",
  "tanggal": "2020-02-26",
  "kata": [
    "kemnaker",
    "awasi",
    "tka",
    "di",
    "meikarta"
  ]
}


H-8. Mini MapReduce: Kasus Kantin

In [45]:
kasir = [
    ["nasi goreng,1", "es teh,2", "mie ayam,1", "es teh,1"],      # kasir A
    ["nasi goreng,2", "es teh,1", "es teh,3", "mie ayam,1"],      # kasir B
    ["mie ayam,2", "nasi goreng,1", "es teh,1", "nasi goreng,1"], # kasir C
]

def map_fn(baris):
    menu, jumlah = baris.split(",")
    return (menu, int(jumlah))

def reduce_fn(menu, daftar_nilai):
    return (menu, sum(daftar_nilai))

# 1) MAP: setiap kasir memproses datanya sendiri
hasil_map = [[map_fn(b) for b in k] for k in kasir]
for nama, h in zip("ABC", hasil_map):
    print("map kasir", nama, "->", h)

# 2) SHUFFLE: kumpulkan nilai berdasarkan key
shuffle = {}
for h in hasil_map:
    for menu, jumlah in h:
        shuffle.setdefault(menu, []).append(jumlah)
print("\nshuffle:", shuffle)

# 3) REDUCE: gabungkan nilai per key
hasil = dict(reduce_fn(m, v) for m, v in shuffle.items())
print("\nreduce:", hasil, "| total porsi:", sum(hasil.values()))

# Combiner: jumlahkan dulu di setiap kasir sebelum dikirim
def combiner(pasangan):
    c = collections.Counter()
    for menu, jumlah in pasangan:
        c[menu] += jumlah
    return list(c.items())

setelah = [combiner(h) for h in hasil_map]
print("\nPasangan dikirim tanpa combiner :", sum(len(h) for h in hasil_map))
print("Pasangan dikirim dengan combiner:", sum(len(h) for h in setelah))

map kasir A -> [('nasi goreng', 1), ('es teh', 2), ('mie ayam', 1), ('es teh', 1)]
map kasir B -> [('nasi goreng', 2), ('es teh', 1), ('es teh', 3), ('mie ayam', 1)]
map kasir C -> [('mie ayam', 2), ('nasi goreng', 1), ('es teh', 1), ('nasi goreng', 1)]

shuffle: {'nasi goreng': [1, 2, 1, 1], 'es teh': [2, 1, 1, 3, 1], 'mie ayam': [1, 1, 2]}

reduce: {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4} | total porsi: 17

Pasangan dikirim tanpa combiner : 12
Pasangan dikirim dengan combiner: 9


H-9. Word Count Terdistribusi: Partisi sebagai Input Split

In [46]:
STOPWORD = set("""yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal
saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya
sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn""".split())

def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())

# Input split = satu file partisi Parquet (satu kategori)
splits = sorted(Path(TERKURASI).rglob("*.parquet"))
print("Jumlah input split:", len(splits))
for p in splits:
    n = len(pd.read_parquet(p, columns=["judul"]))
    print(f"  {p.parent.name:<16} {n:>6,} judul")

def mapper_split(path):
    """Satu mapper memproses satu split dan mengembalikan pasangan (kata, 1)."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    return [(kata, 1) for t in judul for kata in tokenisasi(t)]

hasil_map = [mapper_split(p) for p in splits]
total_pasangan = sum(len(h) for h in hasil_map)

shuffle = collections.defaultdict(list)
for h in hasil_map:
    for kata, n in h:
        shuffle[kata].append(n)
hasil_semua = {k: sum(v) for k, v in shuffle.items()}

print("\nPasangan dari semua mapper:", f"{total_pasangan:,}")
print("Kata unik (key setelah shuffle):", f"{len(hasil_semua):,}")

print("\n10 kata teratas (semua kata):")
for k, n in collections.Counter(hasil_semua).most_common(10):
    print(f"  {k:<10} {n:>7,}")

top = [(k, n) for k, n in collections.Counter(hasil_semua).most_common() if k not in STOPWORD]
print("\n10 kata teratas (tanpa stopword):")
for k, n in top[:10]:
    print(f"  {k:<10} {n:>7,}")

# Cek silang dengan cara biasa (tanpa MapReduce)
cek = collections.Counter(kata for t in bersih["judul"] for kata in tokenisasi(t))
print("\nSama dengan Counter biasa:", dict(cek) == hasil_semua)

Jumlah input split: 9
  kategori=finance 14,141 judul
  kategori=food     4,314 judul
  kategori=health   4,914 judul
  kategori=hot     16,290 judul
  kategori=inet     5,577 judul
  kategori=news    32,321 judul
  kategori=oto      4,375 judul
  kategori=sport    2,433 judul
  kategori=travel   6,453 judul

Pasangan dari semua mapper: 825,601
Kata unik (key setelah shuffle): 32,275

10 kata teratas (semua kata):
  di          27,390
  corona      17,239
  ini         10,124
  yang         8,668
  dan          6,785
  ke           5,592
  tak          5,096
  dari         4,697
  jadi         4,486
  untuk        3,723

10 kata teratas (tanpa stopword):
  corona      17,239
  rp           3,154
  virus        3,075
  kasus        2,743
  polisi       2,732
  positif      2,682
  indonesia    2,504
  warga        2,495
  rumah        2,465
  jokowi       2,317

Sama dengan Counter biasa: True


H-10. Combiner dan Jumlah Data yang Berpindah

In [47]:
from multiprocessing import Pool

def mapper_combiner(path):
    """Mapper + combiner: hitung dulu di sisi mapper."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    c = collections.Counter()
    for t in judul:
        c.update(tokenisasi(t))
    return c

def jalankan(fn, daftar, proses):
    t0 = time.perf_counter()
    if proses == 1:
        hasil = [fn(x) for x in daftar]
    else:
        with Pool(proses) as pool:
            hasil = pool.map(fn, daftar)
    return hasil, time.perf_counter() - t0

print("CPU tersedia:", os.cpu_count())
print(f"{'skenario':<28}{'waktu (s)':>10}{'pasangan dikirim':>20}")
for proses in (1, 2):
    waktu = []
    for _ in range(3):
        h, w = jalankan(mapper_split, splits, proses); waktu.append(w)
    print(f"{proses} proses, tanpa combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}",
          f"{sum(len(x) for x in h):>20,}")
    waktu = []
    for _ in range(3):
        h2, w = jalankan(mapper_combiner, splits, proses); waktu.append(w)
    print(f"{proses} proses, dengan combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}",
          f"{sum(len(x) for x in h2):>20,}")

total = collections.Counter()
for c in h2:
    total.update(c)
print("\nHasil dengan combiner sama persis:", dict(total) == hasil_semua)

CPU tersedia: 2
skenario                     waktu (s)    pasangan dikirim
1 proses, tanpa combiner          0.467              825,601
1 proses, dengan combiner         0.980               74,009
2 proses, tanpa combiner          1.884              825,601
2 proses, dengan combiner         0.675               74,009

Hasil dengan combiner sama persis: True


H-11. Data Skew dan Beban Reducer

In [48]:
R = 4  # jumlah reducer

def partitioner(kata, r=R):
    return zlib.crc32(kata.encode()) % r

def beban(pasangan_per_mapper, r=R):
    b = [0] * r
    for h in pasangan_per_mapper:
        for kata, n in h:
            b[partitioner(kata, r)] += 1
    return b

def tampil(nama, b):
    tot = sum(b)
    print(f"{nama:<22}", "  ".join(f"R{i}={x:>7,} ({x/tot*100:4.1f}%)" for i, x in enumerate(b)))

tampil("tanpa combiner", beban(hasil_map))
tampil("dengan combiner", beban([list(c.items()) for c in h2]))

print("\nBeban reducer terberat dibanding bagian yang adil (tanpa combiner):")
for r in (4, 16, 64):
    b = beban(hasil_map, r)
    print(f"  {r:>3} reducer: terberat {max(b)/sum(b)*100:5.2f}% | bagian adil {100/r:5.2f}%")

print("\nBeban satu key (tanpa combiner):")
tot = sum(hasil_semua.values())
for kata, n in collections.Counter(hasil_semua).most_common(5):
    print(f"  {kata:<8} {n:>7,} pasangan = {n/tot*100:4.1f}% dari total, di reducer R{partitioner(kata)}")

tanpa combiner         R0=240,836 (29.2%)  R1=170,066 (20.6%)  R2=190,727 (23.1%)  R3=223,972 (27.1%)
dengan combiner        R0= 18,301 (24.7%)  R1= 18,267 (24.7%)  R2= 18,530 (25.0%)  R3= 18,911 (25.6%)

Beban reducer terberat dibanding bagian yang adil (tanpa combiner):
    4 reducer: terberat 29.17% | bagian adil 25.00%
   16 reducer: terberat  8.82% | bagian adil  6.25%
   64 reducer: terberat  4.28% | bagian adil  1.56%

Beban satu key (tanpa combiner):
  di        27,390 pasangan =  3.3% dari total, di reducer R0
  corona    17,239 pasangan =  2.1% dari total, di reducer R2
  ini       10,124 pasangan =  1.2% dari total, di reducer R0
  yang       8,668 pasangan =  1.0% dari total, di reducer R3
  dan        6,785 pasangan =  0.8% dari total, di reducer R3


H-12. Hadoop Streaming

In [49]:
Path("masukan.tsv").write_text(
    "\n".join(f"{k}\t{j}" for k, j in zip(bersih["kategori"], bersih["judul"])) + "\n",
    encoding="utf-8")

Path("mapper.py").write_text('''import sys, re
sys.stdin.reconfigure(encoding="utf-8")
for baris in sys.stdin:
    kategori, judul = baris.rstrip("\\n").split("\\t", 1)
    for kata in re.findall(r"[a-z]+", judul.lower()):
        print(f"{kata}\\t1")
''')

Path("reducer.py").write_text('''import sys
kunci_sekarang, total = None, 0
for baris in sys.stdin:
    kunci, nilai = baris.rstrip("\\n").split("\\t")
    if kunci != kunci_sekarang:
        if kunci_sekarang is not None:
            print(f"{kunci_sekarang}\\t{total}")
        kunci_sekarang, total = kunci, 0
    total += int(nilai)
if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\\t{total}")
''')

print("Baris masukan:", sum(1 for _ in open("masukan.tsv", encoding="utf-8")))

Baris masukan: 90818


In [50]:
%%bash
# Uji mapper pada tiga baris pertama, lalu seluruh pipeline: sort = shuffle
head -n 3 masukan.tsv | python3 mapper.py
time (cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py > hasil_pipe.txt)
wc -l hasil_pipe.txt
sort -t$'\t' -k2,2nr hasil_pipe.txt | head -n 5

kemnaker	1
awasi	1
tka	1
di	1
meikarta	1
bni	1
digitalkan	1
bni	1
java	1
jazz	1
terbang	1
ke	1
australia	1
edhy	1
prabowo	1
mau	1
genjot	1
budi	1
daya	1
lobster	1
32275 hasil_pipe.txt
di	27390
corona	17239
ini	10124
yang	8668
dan	6785



real	0m1.904s
user	0m2.057s
sys	0m0.101s


In [51]:
hasil_pipe = {}
for baris in open("hasil_pipe.txt", encoding="utf-8"):
    kata, n = baris.rstrip("\n").split("\t")
    hasil_pipe[kata] = int(n)
print("Hasil pipe sama dengan hasil H-9:", hasil_pipe == hasil_semua)

Hasil pipe sama dengan hasil H-9: True


H-13. Kata Teratas per Kategori, Ekspor ala Sqoop, dan Rangkuman

In [52]:
def mapper_kategori(path, stop=None):
    """Key gabungan (kategori, kata); combiner dipasang di mapper."""
    stop = STOPWORD if stop is None else stop
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in stop:
                c[(kategori, kata)] += 1
    return c

gabung = collections.Counter()
for c in map(mapper_kategori, splits):
    gabung.update(c)

per_kategori = collections.defaultdict(list)
for (kat, kata), n in gabung.items():
    per_kategori[kat].append((n, kata))
for kat in sorted(per_kategori):
    top5 = sorted(per_kategori[kat], reverse=True)[:5]
    print(f"{kat:<8}", ", ".join(f"{k} ({n:,})" for n, k in top5))

finance  corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
food     makanan (699), makan (539), foto (458), galeri (438), enak (358)
health   corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
inet     corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      mobil (1,205), corona (576), motor (432), rp (267), indonesia (256)
sport    corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   corona (1,023), wisata (525), foto (333), dunia (299), turis (284)


In [53]:
import sqlite3
SIAP = Path("siap_pakai"); SIAP.mkdir(exist_ok=True)
baris = [(kat, kata, n) for (kat, kata), n in gabung.items()]
hasil_df = pd.DataFrame(baris, columns=["kategori", "kata", "jumlah"])
hasil_df.to_parquet(SIAP / "kata_per_kategori.parquet", index=False)

# "sqoop export": pindahkan hasil ke database relasional
db = sqlite3.connect(SIAP / "laporan.db")
hasil_df.to_sql("kata_per_kategori", db, if_exists="replace", index=False)
print("Baris diekspor:", db.execute("SELECT COUNT(*) FROM kata_per_kategori").fetchone()[0])
print(db.execute("""SELECT kata, jumlah FROM kata_per_kategori
                    WHERE kategori = 'health' ORDER BY jumlah DESC LIMIT 3""").fetchall())
db.close()

Baris diekspor: 73601
[('corona', 2590), ('virus', 1040), ('pasien', 471)]


Studi Kasus

In [54]:
print(con.sql("""
    SELECT bulan, COUNT(*) AS judul_corona
    FROM berita WHERE lower(judul) LIKE '%corona%'
    GROUP BY bulan ORDER BY bulan""").df().to_string(index=False))
print()
print(con.sql("""
    SELECT kategori,
           COUNT(*) AS total_judul,
           CAST(SUM(CASE WHEN lower(judul) LIKE '%corona%' THEN 1 ELSE 0 END) AS INTEGER) AS judul_corona,
           ROUND(100.0 * SUM(CASE WHEN lower(judul) LIKE '%corona%' THEN 1 ELSE 0 END) / COUNT(*), 1) AS persen
    FROM berita GROUP BY kategori ORDER BY judul_corona DESC""").df().to_string(index=False))

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

kategori  total_judul  judul_corona  persen
    news        32321          7532    23.3
  health         4914          2591    52.7
 finance        14141          2473    17.5
     hot        16290          1663    10.2
  travel         6453          1025    15.9
    inet         5577           847    15.2
     oto         4375           577    13.2
    food         4314           310     7.2
   sport         2433           228     9.4


Latihan 1

In [55]:
TAMBAHAN = {"rp", "ri", "the", "foto", "galeri"}   # ubah sesuai pertimbangan Anda
STOPWORD_BARU = STOPWORD | TAMBAHAN

gabung_baru = collections.Counter()
for p in splits:
    gabung_baru.update(mapper_kategori(p, stop=STOPWORD_BARU))

def top_per_kategori(counter, k=5):
    tmp = collections.defaultdict(list)
    for (kat, kata), n in counter.items():
        tmp[kat].append((n, kata))
    return {kat: [w for _, w in sorted(v, reverse=True)[:k]] for kat, v in tmp.items()}

lama, baru = top_per_kategori(gabung), top_per_kategori(gabung_baru)
print("Kata ditambahkan ke stopword:", sorted(TAMBAHAN))
for kat in sorted(baru):
    print(f"{kat:<8} lama: {lama[kat]}\n{'':<8} baru: {baru[kat]}")

Kata ditambahkan ke stopword: ['foto', 'galeri', 'ri', 'rp', 'the']
finance  lama: ['corona', 'rp', 'ri', 'harga', 'jokowi']
         baru: ['corona', 'harga', 'jokowi', 'mau', 'ekonomi']
food     lama: ['makanan', 'makan', 'foto', 'galeri', 'enak']
         baru: ['makanan', 'makan', 'enak', 'bikin', 'rumah']
health   lama: ['corona', 'virus', 'pasien', 'indonesia', 'meninggal']
         baru: ['corona', 'virus', 'pasien', 'indonesia', 'meninggal']
hot      lama: ['corona', 'anak', 'rumah', 'meninggal', 'the']
         baru: ['corona', 'anak', 'rumah', 'meninggal', 'film']
inet     lama: ['corona', 'virus', 'ponsel', 'bikin', 'indonesia']
         baru: ['corona', 'virus', 'ponsel', 'bikin', 'indonesia']
news     lama: ['corona', 'polisi', 'kasus', 'warga', 'positif']
         baru: ['corona', 'polisi', 'kasus', 'warga', 'positif']
oto      lama: ['mobil', 'corona', 'motor', 'rp', 'indonesia']
         baru: ['mobil', 'corona', 'motor', 'indonesia', 'honda']
sport    lama: ['corona', 

Latihan 2

In [56]:
def mapper_bulan(path):
    d = pd.read_parquet(path, columns=["judul", "bulan"])
    c = collections.Counter()
    for t, b in zip(d["judul"], d["bulan"]):
        for kata in tokenisasi(t):
            if kata not in STOPWORD:
                c[(b, kata)] += 1
    return c

gabung_bulan = collections.Counter()
for p in splits:
    gabung_bulan.update(mapper_bulan(p))

per_bulan = collections.defaultdict(list)
for (b, kata), n in gabung_bulan.items():
    per_bulan[b].append((n, kata))
for b in sorted(per_bulan):
    print(b, ", ".join(f"{k} ({n:,})" for n, k in sorted(per_bulan[b], reverse=True)[:5]))

jan = {k for _, k in sorted(per_bulan["2020-01"], reverse=True)[:15]}
mei = {k for _, k in sorted(per_bulan["2020-05"], reverse=True)[:15]}
print("\nTop-15 hanya di Januari:", sorted(jan - mei))
print("Top-15 hanya di Mei    :", sorted(mei - jan))

2020-01 banjir (1,220), rp (616), polisi (554), jokowi (516), tahun (493)
2020-02 corona (1,076), polisi (630), virus (596), rp (572), banjir (572)
2020-03 corona (6,986), virus (1,340), positif (925), pasien (749), rumah (695)
2020-04 corona (4,914), positif (828), pandemi (706), psbb (689), warga (609)
2020-05 corona (2,935), kasus (640), positif (628), pandemi (596), warga (565)
2020-06 corona (943), normal (471), new (465), kasus (315), rp (265)

Top-15 hanya di Januari: ['banjir', 'baru', 'bikin', 'china', 'jakarta', 'jokowi', 'kpk', 'minta', 'mobil', 'tahun']
Top-15 hanya di Mei    : ['covid', 'lebaran', 'new', 'normal', 'orang', 'pandemi', 'positif', 'psbb', 'rumah', 'warga']


Latihan 3

In [57]:
def mapper_bigram(path):
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    out = []
    for t in judul:
        tok = tokenisasi(t)
        out += [(f"{a} {b}", 1) for a, b in zip(tok, tok[1:])]
    return out

hasil_bigram = [mapper_bigram(p) for p in splits]
tanpa = sum(len(h) for h in hasil_bigram)
comb = [collections.Counter(k for k, _ in h) for h in hasil_bigram]
dengan = sum(len(c) for c in comb)
total_bg = collections.Counter()
for c in comb:
    total_bg.update(c)

print(f"Pasangan tanpa combiner : {tanpa:,}")
print(f"Pasangan dengan combiner: {dengan:,}  (turun {tanpa/dengan:.1f}x)")
print("Bigram 'virus corona'   :", f"{total_bg['virus corona']:,}")
print("10 bigram teratas       :", total_bg.most_common(10))

Pasangan tanpa combiner : 734,783
Pasangan dengan combiner: 411,217  (turun 1.8x)
Bigram 'virus corona'   : 2,915
10 bigram teratas       : [('virus corona', 2915), ('corona di', 2178), ('positif corona', 1804), ('di tengah', 1258), ('pandemi corona', 1075), ('di rumah', 1047), ('new normal', 917), ('di indonesia', 871), ('rp juta', 700), ('hari ini', 627)]


Latihan 4

In [58]:
print(f"{'R':>4} {'terberat (%)':>14} {'bagian adil (%)':>17} {'rasio':>7}")
for r in (2, 8, 32):
    b = beban(hasil_map, r)
    terberat, adil = max(b) / sum(b) * 100, 100 / r
    print(f"{r:>4} {terberat:>14.2f} {adil:>17.2f} {terberat/adil:>7.2f}")

   R   terberat (%)   bagian adil (%)   rasio
   2          52.27             50.00    1.05
   8          16.83             12.50    1.35
  32           5.82              3.12    1.86


Latihan 5

In [59]:
t0 = time.perf_counter()
n_judul = len(list(crawl_iter(BASE, range(1, 51), delay=0.05)))
t50 = time.perf_counter() - t0
overhead = (t50 - 50 * 0.05) / 50          # waktu kerja nyata per halaman (di luar jeda)
est_911_jeda1 = 911 * (overhead + 1.0)
print(f"50 halaman, jeda 0,05 dtk: {t50:.2f} dtk ({n_judul} judul)")
print(f"Estimasi 911 halaman, jeda 1 dtk: {est_911_jeda1:.0f} dtk = {est_911_jeda1/60:.1f} menit")

50 halaman, jeda 0,05 dtk: 5.90 dtk (5000 judul)
Estimasi 911 halaman, jeda 1 dtk: 973 dtk = 16.2 menit


Latihan 6

In [60]:
JALANKAN_LATIHAN_6 = False
SITEMAP_LAIN = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://sport.detik.com/basket/sitemap_news.xml",   # <- ganti dengan alamat lain dari sitemap.xml
]

def crawl_custom(daftar_sitemap):
    sesi = requests.Session(); sesi.headers["User-Agent"] = UA_DETIK
    hasil = []
    for sm in daftar_sitemap:
        if len(hasil) >= BATAS_TOTAL:
            break
        hasil += parse_sitemap_news(ambil_xml(sm, sesi))[:BATAS_PER_BAGIAN]
    return hasil[:BATAS_TOTAL]

try:
    data6 = crawl_custom(SITEMAP_LAIN) if JALANKAN_LATIHAN_6 else sampel
except Exception as e:
    data6 = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if data6:
    d6 = pd.DataFrame(data6)
    print("Kategori:", d6["kategori"].value_counts().to_dict())
    for nama, lulus in aturan_kualitas(d6).items():
        print(f"{nama:<24} gagal: {(~lulus).sum():>3} dari {len(d6)}")
else:
    print("Tidak ada data crawl (belum ada internet ke detik.com / crawl gagal).")

Kategori: {'news': 7, '': 7, 'sport': 6}
judul tidak kosong       gagal:   0 dari 20
judul 3-30 kata          gagal:   0 dari 20
kategori dikenal         gagal:   7 dari 20
tanggal valid            gagal:   0 dari 20
url artikel valid        gagal:   7 dari 20
judul tidak duplikat     gagal:   0 dari 20


Latihan 7

In [61]:
import itertools

# Sumber data: hasil crawl H-4 bila ada; bila tidak, pakai CONTOH_XML agar tetap bisa dijalankan
berkas = Path("contoh_crawl/detik_sitemap.jsonl")
if berkas.exists():
    data7 = [json.loads(l) for l in open(berkas, encoding="utf-8")]
else:
    data7 = parse_sitemap_news(CONTOH_XML)

def mapper_kunci(rec):
    for k in rec.get("kata_kunci", "").split(","):
        k = k.strip().lower()
        if k:
            yield (k, 1)

def reducer_kunci(pasangan_terurut):
    for k, grup in itertools.groupby(pasangan_terurut, key=lambda p: p[0]):
        yield k, sum(n for _, n in grup)

pasangan = [p for rec in data7 for p in mapper_kunci(rec)]   # MAP
pasangan.sort()                                               # SHUFFLE (sort)
hasil_kunci = sorted(reducer_kunci(pasangan), key=lambda x: -x[1])  # REDUCE
print("Kata kunci teratas:", hasil_kunci[:10])

Kata kunci teratas: [('ibl all indonesian', 3), ('ibl all indonesian 2026', 3), ('ahrie sonta', 2), ('lipsus smartfren', 2), ('operator', 2), ('polri', 2), ('telkomsel', 2), ('4g', 1), ('alfamidi', 1), ('alumni dan suporter', 1)]


Tugas Mandiri

In [62]:
def id_dokumen(id_asli, judul):
    return int(id_asli) if pd.notna(id_asli) else -zlib.crc32(judul.encode())

def mapper_index(path):
    d = pd.read_parquet(path, columns=["judul", "id"])
    out = []
    for judul, i in zip(d["judul"], d["id"]):
        did = id_dokumen(i, judul)
        for kata in set(tokenisasi(judul)) - STOPWORD:   # set: tiap id hanya sekali per kata
            out.append((kata, did))
    return out

def reducer_index(pasangan_per_mapper):
    idx = collections.defaultdict(list)                  # SHUFFLE: kelompokkan per kata
    for h in pasangan_per_mapper:
        for kata, did in h:
            idx[kata].append(did)
    # REDUCE: daftar id unik; id asli (positif) dulu, lalu id pengganti
    return {k: sorted(set(v), key=lambda x: (x < 0, abs(x))) for k, v in idx.items()}

inverted = reducer_index([mapper_index(p) for p in splits])
print("Jumlah kata dalam indeks:", f"{len(inverted):,}")
for kata in ("corona", "virus", "polisi"):
    print(f"{kata:<8} {len(inverted[kata]):>6,} dokumen | 5 id pertama: {inverted[kata][:5]}")

n_pengganti = sum(1 for v in inverted["corona"] if v < 0)
print("\nDokumen 'corona' dengan id pengganti (inet):", n_pengganti)

pd.DataFrame({"kata": list(inverted), "daftar_id": list(inverted.values())}) \
  .to_parquet(SIAP / "inverted_index.parquet", index=False)
print("Disimpan:", SIAP / "inverted_index.parquet")

Jumlah kata dalam indeks: 32,228
corona   17,228 dokumen | 5 id pertama: [4856495, 4856598, 4868317, 4868545, 4869163]
virus     3,067 dokumen | 5 id pertama: [4856495, 4856598, 4860201, 4865102, 4865341]
polisi    2,724 dokumen | 5 id pertama: [4842273, 4842280, 4842541, 4842612, 4843641]

Dokumen 'corona' dengan id pengganti (inet): 845
Disimpan: siap_pakai/inverted_index.parquet
